In [13]:
# project root to src dir setup
import sys
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Success! Project root added: {parent_dir}")
print("Available folders here:", os.listdir(parent_dir))


Success! Project root added: c:\Users\arsal\OneDrive\Desktop\Study Stuff\Projects\AIML\Data Analysis\solar-performance-intelligence
Available folders here: ['.git', '.gitignore', 'data', 'LICENSE', 'README.md', 'scripts', 'solar-power-generation-and-energy-consumption-data.zip', 'src']


In [21]:
from src.data.loader import DataLoader
from src.data.transformer import DataTransformer
from src.config.settings import RAW_DIR
from src.utils.profiler import profile_dataset

In [18]:
loader = DataLoader(RAW_DIR)
datasets = loader.load_all()

In [25]:
generation = (
        DataTransformer
        .transform_generation(
            datasets["generation"]
        )
        # .pipe(DataTransformer.optimize_dataframe)
    )
    
weather = (
    DataTransformer
    .transform_weather(
        datasets["weather"]
    )
    # .pipe(DataTransformer.optimize_dataframe)
)

irradiance = (
    DataTransformer
    .transform_irradiance(
        datasets["irradiance"]
    )
    # .pipe(DataTransformer.optimize_dataframe)
)

sites = (
    DataTransformer
    .transform_sites(
        datasets["sites"]
    )
    # .pipe(DataTransformer.optimize_dataframe)
)

# Data Profiling
I made a extensive report with shape, nulls and nulls pct, duplicate rows, and memory consumption done by it

In [26]:
for name, df in datasets.items():
    profile_dataset(name=name, df=df)


> GENERATION
Shape:  (2731946, 4)
Memory(Mb):  260.54
Duplicate Rows:  0

Null Report:
                  missing_count  missing_pct
SolarGeneration        1536301    56.234677
CampusKey                    0     0.000000
SiteKey                      0     0.000000
Timestamp                    0     0.000000

> WEATHER
Shape:  (371769, 8)
Memory(Mb):  46.8
Duplicate Rows:  0

Null Report:
                      missing_count  missing_pct
WindSpeed                   162890    43.814842
WindDirection               162890    43.814842
AirTemperature              107113    28.811708
ApparentTemperature         107113    28.811708
RelativeHumidity            107113    28.811708
DewPointTemperature         107113    28.811708
CampusKey                        0     0.000000
Timestamp                        0     0.000000

> IRRADIANCE
Shape:  (102360, 4)
Memory(Mb):  15.31
Duplicate Rows:  0

Null Report:
                missing_count  missing_pct
CloudOpacity               0          0.0
Ghi  

The below was done in order to understand weather there was some due to power outages or some timely shutting off of the devices

In [24]:
import pandas as pd

generation = pd.read_csv(RAW_DIR / "Solar_Energy_Generation.csv")

generation["Timestamp"] = pd.to_datetime(generation["Timestamp"])

nulls_by_hour = generation[
    generation["SolarGeneration"].isna()
]["Timestamp"].dt.hour.value_counts().sort_index()

print(nulls_by_hour)

print("\n\n\n" ,generation.groupby(
    "SiteKey"
)["SolarGeneration"].apply(
    lambda x:
    x.isnull().mean() * 100
).sort_values(ascending=False))

Timestamp
0     114050
1     114447
2     114325
3     114464
4     114438
5     114297
6     103891
7      56296
8      14685
9       8127
10      7294
11      8368
12     13793
13     17241
14     13818
15      8040
16      8342
17     29713
18     53961
19     67187
20    100491
21    113143
22    113042
23    112848
Name: count, dtype: int64



 SiteKey
7     74.543300
5     61.942179
9     60.924873
35    59.736378
29    59.141102
28    59.111230
13    59.083089
4     58.748279
16    58.520075
34    57.996847
21    56.244164
32    56.188134
39    56.105622
14    55.998687
26    55.984250
27    55.955141
17    55.702546
30    55.640545
36    55.511112
42    55.455818
20    55.100230
22    55.097118
37    55.087779
40    55.006848
38    54.916578
31    54.838760
33    54.728257
23    54.603066
24    54.569508
15    54.555521
12    54.502704
19    54.320595
25    54.058307
10    54.048841
3     53.809302
18    53.760329
2     53.649189
41    53.084381
8     52.969720
1     52.630517


# Some quick inference
## Irradiance
This has 0% missing data so it means for now we can trust it

## Sites
has 42 rows of metadata about the sites, with missing columns being Panel, Inverter, Optimizers, Metric, Number of Panels, kWp with somewhere as shown above in the report by profile_dataset() function output
Features like Panel, Inverter, Optimizer, and Metric isnt really of use for 

## Weather
the columns Tempearture, Humidity, Dewpoint being approx 29% missing, and Wind being around 44% missing makes it so we can infer that there is either sensor outages, or a partial collection period only.

## 

In [ ]:
null_pct_by_site = sites_clean.groupby('CampusKey').apply(lambda group: group.isna().mean() * 100)

print(null_pct_by_site[['kWp', 'Number of panels']])

                  kWp  Number of panels
CampusKey                              
1            7.407407          7.407407
2          100.000000        100.000000
3          100.000000        100.000000
4          100.000000        100.000000
5          100.000000        100.000000


In [36]:
sites_clean.groupby(sites_clean['SiteKey']).agg(lambda x: x.isna().mean() * 100)

,CampusKey,kWp,Number of panels,lat,Lon
SiteKey,,,,,
1,0.0,100.0,100.0,0.0,0.0
2,0.0,100.0,100.0,0.0,0.0
3,0.0,100.0,100.0,0.0,0.0
4,0.0,100.0,100.0,0.0,0.0
5,0.0,100.0,100.0,0.0,0.0
6,0.0,100.0,100.0,0.0,0.0
7,0.0,100.0,100.0,0.0,0.0
8,0.0,100.0,100.0,0.0,0.0
9,0.0,100.0,100.0,0.0,0.0
